## 1. Bronze Visit Schema

**Question**

What is the structure and data type definition of the Bronze EDC visit dataset?

**Purpose**

Understand the raw Bronze visit structure before designing Silver normalization, type handling, data-quality rules, and visit processing.

In [0]:
display(
    spark.sql("""
        DESCRIBE TABLE
        clinical_trial_intelligence.bronze.edc_visits
    """)
)

### Result
The Bronze visit dataset contains the following business fields:

- `visit_id`
- `subject_id`
- `study_id`
- `site_id`
- `visit_name`
- `visit_number`
- `visit_date`
- `visit_status`
- `days_from_baseline`

The dataset also retains Bronze operational and lineage metadata:

- `_rescued_data`
- `_source_file`
- `_source_file_name`
- `_source_file_modification_ts`
- `_ingestion_ts`
- `_ingestion_date`

The Bronze ingestion layer has already inferred the expected data types for the principal visit attributes. In particular, `visit_number` and `days_from_baseline` are integers, while `visit_date` is stored as a date.

### Design conclusion

The Bronze visit schema contains the core identifiers, visit lifecycle attributes, and lineage metadata required for Silver processing.

Silver visit processing should preserve the Bronze lineage columns, standardize identifier and categorical fields, validate visit-level business rules and relationships, and determine the appropriate handling of invalid records based on subsequent exploration.

The `_rescued_data` field should also be investigated before finalizing the Silver contract because populated rescued data can indicate schema mismatch or data that could not be parsed into the expected columns.

## 2. Sample Bronze Visit Records

### Question

What do representative Bronze visit records look like before Silver transformation?

### Purpose

Inspect actual source values, identifier formats, visit names, statuses, dates, baseline offsets, missing values, and ingestion metadata before defining Silver business rules.

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM clinical_trial_intelligence.bronze.edc_visits
        LIMIT 50
    """)
)

### Result

The sampled Bronze records show a structured visit sequence associated with each clinical trial subject.

Observed visit names include `SCREENING`, `BASELINE`, `WEEK_2`, `WEEK_4`, `WEEK_8`, `WEEK_12`, `WEEK_24`, and `EOT`, with corresponding visit numbers.

The sample contains visit statuses including `COMPLETED`, `MISSED`, and `RESCHEDULED`.

`days_from_baseline` contains both negative and positive values. Negative values occur for pre-baseline visits such as `SCREENING`; however, some sampled `BASELINE` records also have small non-zero offsets. Therefore, this field should not be assumed to equal zero for every BASELINE record without further validation.

The sampled records contain populated subject, study, site, visit, and date identifiers, and `_rescued_data` is NULL for the inspected records.

All displayed records originate from `visits_20260825.csv`, while Bronze lineage and ingestion metadata are retained.

### Design conclusion

The sample indicates that the visit feed represents subject-level clinical visit events with an apparent visit sequence and multiple visit statuses.

However, the sample alone is insufficient to establish Silver data-quality rules. Full-dataset profiling is required to determine identifier completeness, duplicate visit keys, categorical value domains, relationship integrity, visit sequencing, and whether records change across source files.

Therefore, no additional Silver transformation or rejection rule is introduced from this sample alone.

## 3. Bronze Visit Dataset Profile

### Question

What is the overall size, business-key completeness, source-file coverage, and rescued-data condition of the Bronze visit dataset?

### Purpose

Establish the baseline Bronze visit population and identify structural data-quality issues before investigating visit-level business rules and CDC behavior.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT visit_id) AS distinct_visit_ids,

    COUNT_IF(
        visit_id IS NULL
        OR TRIM(visit_id) = ''
    ) AS missing_visit_id,

    COUNT_IF(
        subject_id IS NULL
        OR TRIM(subject_id) = ''
    ) AS missing_subject_id,

    COUNT_IF(
        study_id IS NULL
        OR TRIM(study_id) = ''
    ) AS missing_study_id,

    COUNT_IF(
        site_id IS NULL
        OR TRIM(site_id) = ''
    ) AS missing_site_id,

    COUNT_IF(visit_date IS NULL)
        AS missing_visit_date,

    COUNT_IF(
        _rescued_data IS NOT NULL
        AND TRIM(_rescued_data) <> ''
    ) AS rescued_data_rows,

    COUNT(DISTINCT _source_file_name)
        AS source_file_count

FROM clinical_trial_intelligence.bronze.edc_visits;

### Result

The Bronze visit dataset contains 18,262 records across 10 source files, with 18,262 distinct `visit_id` values. Therefore, no duplicate visit identifiers are present at the Bronze level.

The primary visit business key is complete, with zero missing `visit_id` values. `study_id` and `site_id` are also complete.

Two structural data-quality issues are present:

- 218 records have a missing `subject_id`.
- 102 records have a missing `visit_date`.

No populated `_rescued_data` records were identified, indicating that the Bronze visit records were parsed into the expected schema without rescued fields.

### Design conclusion

`visit_id` can be treated as the candidate business key for Silver visit processing because it is complete and unique across the current Bronze dataset.

Missing `subject_id` and missing `visit_date` are confirmed data-quality conditions and must be evaluated during Silver processing rather than treated only as warning-level observations.

No additional handling is currently required for `_rescued_data`.

The 10 source files must next be investigated to determine whether they represent independent visit events, incremental changes, or repeated snapshots. This is necessary before deciding whether the Silver visit target requires CDC/history handling or simple append/validated processing.

## 4. Visit Source-File Inventory

### Question

How are Bronze visit records distributed across source files, and what does the file-level population indicate about the visit feed structure?

### Purpose

Inspect row counts, distinct visits, distinct subjects, and ingestion timing across source files to determine whether the visit feed behaves as independent event files, incremental change files, or repeated snapshots.

In [0]:
%sql

SELECT
    _source_file_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT visit_id) AS distinct_visits,
    COUNT(DISTINCT subject_id) AS distinct_subjects,
    MIN(_ingestion_ts) AS first_ingestion,
    MAX(_ingestion_ts) AS last_ingestion

FROM clinical_trial_intelligence.bronze.edc_visits

GROUP BY _source_file_name

ORDER BY _source_file_name;

### Result

The Bronze visit feed consists of 10 source files. The initial file (`visits_20260825.csv`) contains 17,725 visit records, while subsequent files contain substantially smaller record volumes ranging from 24 to 169 rows.

Within every source file, the number of rows equals the number of distinct `visit_id` values, indicating that duplicate visit identifiers do not occur within individual source files.

All source files were ingested during the same Bronze pipeline execution. Therefore, `_ingestion_ts` cannot be used to determine the logical ordering of the source files.

The filename date provides the available logical ordering of the visit source files.

## 5. Cross-File Visit Investigation

### Question

Does the same `visit_id` appear across multiple source files?

### Purpose

Determine whether later visit files contain only new visit events or also contain updated versions of previously received visits. This determines whether Silver Visits can use simple incremental processing or requires CDC/history handling.

In [0]:
%sql

SELECT
    TRIM(visit_id) AS visit_id,
    COUNT(*) AS version_count,
    COUNT(DISTINCT _source_file_name) AS source_file_count,
    MIN(_source_file_name) AS first_source_file,
    MAX(_source_file_name) AS latest_source_file,
    COLLECT_SET(_source_file_name) AS source_files

FROM clinical_trial_intelligence.bronze.edc_visits

WHERE visit_id IS NOT NULL
  AND TRIM(visit_id) <> ''

GROUP BY
    TRIM(visit_id)

HAVING COUNT(*) > 1

ORDER BY
    version_count DESC,
    visit_id

LIMIT 50;

### Result

No `visit_id` appears more than once across the Bronze visit source files.

Therefore, the available visit feed behaves as an append-only event dataset rather than a snapshot or CDC dataset. Each `visit_id` represents a distinct clinical visit event.

Unlike the Subjects dataset, Visits does not require SCD Type 2 processing or source-snapshot sequencing.

Silver Visits can therefore be processed incrementally from Bronze using `spark.readStream.table(...)`, with `visit_id` as the event-level business key.

## 6. Missing Critical Fields Investigation

### Question

What is the nature of Bronze visit records with missing `subject_id` or `visit_date`?

### Purpose

Determine whether records missing critical visit attributes should remain in the trusted Silver Visits table or be quarantined.

In [0]:
%sql

SELECT
    CASE
        WHEN subject_id IS NULL OR TRIM(subject_id) = ''
            THEN 'missing_subject_id'

        WHEN visit_date IS NULL
            THEN 'missing_visit_date'

        ELSE 'other'
    END AS issue_type,

    COUNT(*) AS affected_rows,
    COUNT(DISTINCT visit_id) AS distinct_visits,
    COUNT(DISTINCT _source_file_name) AS affected_source_files

FROM clinical_trial_intelligence.bronze.edc_visits

WHERE subject_id IS NULL
   OR TRIM(subject_id) = ''
   OR visit_date IS NULL

GROUP BY
    CASE
        WHEN subject_id IS NULL OR TRIM(subject_id) = ''
            THEN 'missing_subject_id'

        WHEN visit_date IS NULL
            THEN 'missing_visit_date'

        ELSE 'other'
    END

ORDER BY affected_rows DESC;

### Result

The investigation identified two critical data-quality conditions:

- 218 distinct visit records have a missing `subject_id`, affecting 6 source files.
- 102 distinct visit records have a missing `visit_date`, affecting 2 source files.

Each affected row represents a distinct `visit_id`, indicating that these failures are source-data quality issues rather than duplicate ingestion artifacts.

Both fields are required for the trusted Silver visit dataset. Records without a `subject_id` cannot be reliably associated with a clinical subject, while records without a `visit_date` cannot support reliable temporal visit analysis.

Therefore, these records should be preserved for auditability but excluded from the trusted Silver `visits` dataset and routed to a quarantine dataset with an explicit failure reason.

## 7. Overlapping Critical DQ Failures

### Question

Do any visit records have both `subject_id` and `visit_date` missing?

### Purpose

Determine the exact number of unique invalid visit records and ensure quarantine failure reasons accurately represent multiple simultaneous data-quality failures.

In [0]:
%sql

SELECT
    COUNT(*) AS both_missing_rows
FROM clinical_trial_intelligence.bronze.edc_visits
WHERE (subject_id IS NULL OR TRIM(subject_id) = '')
  AND visit_date IS NULL;

### Result

No Bronze visit record has both `subject_id` and `visit_date` missing simultaneously.

Therefore, the two critical data-quality failure populations are mutually exclusive:

- 218 records fail because `subject_id` is missing.
- 102 records fail because `visit_date` is missing.
- Total unique records requiring quarantine = 320.

These records should be preserved in the quarantine layer while being excluded from the trusted Silver `visits` dataset.

In [0]:
%sql

SELECT
    COALESCE(
        UPPER(TRIM(visit_status)),
        '<NULL>'
    ) AS visit_status,

    COUNT(*) AS missing_visit_date_rows,

    COUNT(DISTINCT visit_id) AS distinct_visits,

    COUNT(DISTINCT _source_file_name) AS source_file_count

FROM clinical_trial_intelligence.bronze.edc_visits

WHERE visit_date IS NULL

GROUP BY
    COALESCE(
        UPPER(TRIM(visit_status)),
        '<NULL>'
    )

ORDER BY
    missing_visit_date_rows DESC;

Grain:
1 row = 1 subject visit

Business key:
visit_id

Rows:
18,262

Distinct visit_id:
18,262

Duplicate visit_id:
0

Duplicate (subject_id, visit_number):
0

Ingestion pattern:
append-only based on currently observed files

Rescued rows:
0

Status problems:
?        → UNKNOWN
complete → COMPLETED
DONE     → COMPLETED

Missing values:
subject_id → 218
visit_date → 102
study_id   → 0
site_id    → 0
visit_id   → 0